# 01.4 Dataset and DataLoader

This notebook answers a very practical question: how data is fed into a model batch by batch during training.

Key concepts:

- dataset
- sample
- batch
- shuffle
- data loader
- custom dataset

If this notebook is not clear, later training loops often become messy.


## Learning Goals

After this notebook, you should be able to:

1. Understand the different roles of `Dataset` and `DataLoader`.
2. Implement a minimal custom `Dataset`.
3. Use `DataLoader` to create batches.
4. Understand the roles of `batch_size` and `shuffle`.
5. Read the shapes inside a batch.
6. Prepare the data input interface for later training loops.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

## The Difference Between `Dataset` and `DataLoader`

The simplest way to think about them:

- defines what the i-th sample is
- defines how samples are batched and iterated

You can also remember it like this:

- `Dataset` is responsible for content
- `DataLoader` is responsible for organization and delivery

## A Quick Start with Built-in `TensorDataset`

Before implementing a custom dataset, we start with `TensorDataset` to feel the minimal workflow.


In [ ]:
X = torch.tensor(
    [
        [1.0, 0.5],
        [2.0, 1.0],
        [3.0, 1.5],
        [4.0, 2.0],
        [5.0, 2.5],
        [6.0, 3.0],
    ],
    dtype=torch.float32,
)
y = torch.tensor([0, 0, 0, 1, 1, 1], dtype=torch.long)

dataset = TensorDataset(X, y)

print("number of samples / number of samples:", len(dataset))
print("sample 0 / sample 0:", dataset[0])
print("sample 3 / sample 3:", dataset[3])

In [ ]:
loader = DataLoader(dataset, batch_size=2, shuffle=False)

for batch_idx, (xb, yb) in enumerate(loader):
    print(f"batch {batch_idx}")
    print("xb =\n", xb)
    print("yb =", yb)
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print()

The most important thing here is the shapes:

- one sample feature shape: `(2,)`
- one batch feature shape: `(batch_size, 2)`
- one batch label shape: `(batch_size,)`

That is, `DataLoader` automatically adds the batch dimension in front.


In [ ]:
# Exercise 1
# Create a DataLoader from the dataset above.
#Requirements / Requirements:
# 1. batch_size=3
# 2. shuffle=False

# loader_ex =
# for xb, yb in loader_ex:
#     print(xb.shape, yb.shape)

In [ ]:
# Exercise 1 Reference Solution

loader_ex = DataLoader(dataset, batch_size=3, shuffle=False)
for xb, yb in loader_ex:
    print(xb.shape, yb.shape)

## Building a Custom `Dataset`

In real projects, you usually need to define how data is read yourself, instead of only using `TensorDataset`.

A minimal custom `Dataset` only needs two things:

- `__len__`
- `__getitem__`

In [ ]:
class SimpleTabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

        assert len(self.features) == len(self.labels), (
            "features and labels must have the same length / features and labels must have the same length"
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        x = self.features[index]
        y = self.labels[index]
        return x, y


features = [
    [1.0, 0.2],
    [2.0, 0.4],
    [3.0, 0.7],
    [4.0, 0.9],
]
labels = [0, 0, 1, 1]

custom_ds = SimpleTabularDataset(features, labels)
print("len(custom_ds) =", len(custom_ds))
print("custom_ds[2] =", custom_ds[2])

This class does a few important things:

- converts inputs to tensors
- ensures features and labels have the same length
- returns one sample per index

In [ ]:
# Exercise 2
# Implement a Dataset that returns a dict:
# {"features": x, "label": y}

class DictDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        # TODO
        pass

    def __getitem__(self, index):
        # TODO
        pass


# ds = DictDataset(features, labels)
# print(len(ds))
# print(ds[0])

In [ ]:
# Exercise 2 Reference Solution

class DictDatasetSolution(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return {
            "features": self.features[index],
            "label": self.labels[index],
        }


ds = DictDatasetSolution(features, labels)
print(len(ds))
print(ds[0])

## `batch_size` and `shuffle`

These two arguments appear in almost every training workflow.

- how many samples go into the model at once
- whether to shuffle sample order each epoch

Training sets usually use `shuffle=True`, while validation and test sets usually use `False`.


In [ ]:
torch.manual_seed(42)

loader_no_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=False)
loader_shuffle = DataLoader(custom_ds, batch_size=2, shuffle=True)

print("no shuffle / no shuffle")
for xb, yb in loader_no_shuffle:
    print(xb[:, 0], yb)

print()
print("shuffle / shuffle")
for xb, yb in loader_shuffle:
    print(xb[:, 0], yb)

## What a Batch Actually Looks Like

Understanding batch shapes is foundational for reading model inputs and outputs later.

In general:

- single-sample features: `(num_features,)`
- batch features: `(batch_size, num_features)`
- single label / single label: `()` or `(1,)`
- batch labels: `(batch_size,)`

In [ ]:
loader = DataLoader(custom_ds, batch_size=3, shuffle=False)
xb, yb = next(iter(loader))

print("xb =\n", xb)
print("yb =", yb)
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

In [ ]:
# Exercise 3
# Create a DataLoader from custom_ds with batch_size=4.
#Take the first batch and print:
# Take the first batch and print:
# 1. xb.shape
# 2. yb.shape
# 3. xb[0]
# 4. yb[0]

# loader_big =
# xb, yb = next(iter(loader_big))
# print(xb.shape)
# print(yb.shape)
# print(xb[0])
# print(yb[0])

In [ ]:
# Exercise 3 Reference Solution

loader_big = DataLoader(custom_ds, batch_size=4, shuffle=False)
xb, yb = next(iter(loader_big))
print(xb.shape)
print(yb.shape)
print(xb[0])
print(yb[0])

## Integrated Mini Example

Now we combine `Dataset`, `DataLoader`, and batch iteration into one small example.


In [ ]:
train_loader = DataLoader(custom_ds, batch_size=2, shuffle=True)

for step, (xb, yb) in enumerate(train_loader):
    batch_mean = xb.mean(dim=0)
    print(f"step={step}")
    print("xb.shape =", xb.shape)
    print("yb.shape =", yb.shape)
    print("batch mean / batch mean =", batch_mean)
    print()

## Summary

The most important distinction in this notebook is:

- `Dataset` defines samples
- `DataLoader` organizes batches

You should now be able to answer:

1. Why does a minimal `Dataset` only need `__len__` and `__getitem__`?
2. How does `batch_size` affect batch shapes?
3. Why do training sets often use `shuffle=True`?
4. Why do models usually receive batches rather than single samples?

Suggested next step:

- Move to the `nn.Module` notebook and start defining actual network structures.